# Modelo preditivo - baseline vs regressao

Objetivo: prever o faturamento do proximo mes. Duas abordagens comparadas no mesmo
periodo de teste:

1. **Baseline ingenuo**: media movel dos 3 meses anteriores. Sem ele, o MAE da
   regressao e um numero solto - a comparacao com uma referencia burra e o que diz
   se o modelo aprendeu alguma coisa.
2. **Regressao linear**: tendencia (indice do mes) + dummies de mes para capturar
   sazonalidade.

Split temporal: os ultimos 6 meses (jan-jun/2026) ficam de fora do treino e viram
teste. Metricas: MAE, MAPE e R2. O resultado materializado em `dw.previsao_mensal` e
`dw.metricas_modelo` sai do `src/modelo/treinar.py`, que replica este notebook - aqui
e onde a escolha se justifica.

In [1]:
import os

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sqlalchemy import create_engine

load_dotenv("../.env")
# o driver e psycopg3; SQLAlchemy precisa do esquema explicito na URL
url = os.environ["DATABASE_URL"].replace("postgresql://", "postgresql+psycopg://", 1)
eng = create_engine(url)

In [2]:
serie = pd.read_sql("""
    SELECT ano_mes, ROUND(SUM(valor_total), 2) AS faturamento
    FROM dw.vw_vendas
    WHERE status_pedido IN ('Entregue', 'Enviado')
    GROUP BY ano_mes ORDER BY ano_mes
""", eng)
serie["faturamento"] = serie["faturamento"].astype(float)
serie

,ano_mes,faturamento
0,2024-07,187636.21
1,2024-08,202990.18
2,2024-09,224241.17
3,2024-10,195820.24
4,2024-11,335323.65
5,2024-12,288509.58
6,2025-01,185879.29
7,2025-02,179800.82
8,2025-03,164442.27
9,2025-04,208585.90


24 pontos, jul/2024 a jun/2026 - mesma serie da EDA. Dois padroes que o modelo
precisa capturar: o salto de novembro nos dois anos (Black Friday: 335k em 2024,
500k em 2025, contra ~200-300k de mes tipico) e o crescimento de nivel entre o
primeiro e o segundo ano.

24 observacoes e pouco para serie temporal seria (ARIMA, Prophet etc. estariam
superdimensionados e sem dado para validar). Regressao com tendencia + dummies e o
teto honesto de complexidade para esse volume - e e o que o enunciado pede.

In [9]:
df = serie.copy()
df["t"] = np.arange(len(df))
df["mes"] = df["ano_mes"].str[5:].astype(int)

X = pd.get_dummies(df[["t", "mes"]], columns=["mes"], prefix="m", drop_first=True)
y = df["faturamento"]

N_TESTE = 6
X_treino, X_teste = X.iloc[:-N_TESTE], X.iloc[-N_TESTE:]
y_treino, y_teste = y.iloc[:-N_TESTE], y.iloc[-N_TESTE:]
print(f"treino: {df['ano_mes'].iloc[0]} a {df['ano_mes'].iloc[-N_TESTE-1]} ({len(X_treino)} meses)")
print(f"teste:  {df['ano_mes'].iloc[-N_TESTE]} a {df['ano_mes'].iloc[-1]} ({len(X_teste)} meses)")

treino: 2024-07 a 2025-12 (18 meses)
teste:  2026-01 a 2026-06 (6 meses)


Por que split temporal e nao aleatorio: previsao e um problema de futuro. Um split
aleatorio deixaria o modelo "ver" meses posteriores aos que ele tenta prever -
vazamento de informacao que infla as metricas e nao corresponde a nenhum uso real.
O teste simula exatamente o cenario de producao: treinar ate dez/2025 e prever 2026.

Custo assumido: com 18 meses de treino, as dummies de jan-jun sao estimadas com UMA
observacao cada (a de 2025). Na pratica, a previsao desses meses vira "mesmo mes do
ano passado + tendencia" - um sazonal ingenuo interpretavel. Com 24 pontos nao ha
como fazer melhor sem inventar dado.

In [4]:
# baseline: media movel dos 3 meses anteriores (shift(1) garante que o mes
# previsto nunca entra na propria media)
df["mm3"] = df["faturamento"].shift(1).rolling(3).mean()
pred_baseline = df["mm3"].iloc[-N_TESTE:]
pd.DataFrame({"ano_mes": df["ano_mes"].iloc[-N_TESTE:], "real": y_teste.round(0),
              "baseline_mm3": pred_baseline.round(0)})

,ano_mes,real,baseline_mm3
18,2026-01,293451.0,354766.0
19,2026-02,330627.0,374656.0
20,2026-03,297147.0,317973.0
21,2026-04,193269.0,307075.0
22,2026-05,269745.0,273681.0
23,2026-06,287209.0,253387.0


In [5]:
reg = LinearRegression().fit(X_treino, y_treino)
pred_reg = pd.Series(reg.predict(X_teste), index=y_teste.index)
pd.DataFrame({"ano_mes": df["ano_mes"].iloc[-N_TESTE:], "real": y_teste.round(0),
              "regressao": pred_reg.round(0)})

,ano_mes,real,regressao
18,2026-01,293451.0,255841.0
19,2026-02,330627.0,249763.0
20,2026-03,297147.0,234404.0
21,2026-04,193269.0,278548.0
22,2026-05,269745.0,245341.0
23,2026-06,287209.0,267005.0


In [6]:
def metricas(y_real, y_pred):
    return {
        "mae": mean_absolute_error(y_real, y_pred),
        "mape": float(np.mean(np.abs((y_real - y_pred) / y_real)) * 100),
        "r2": r2_score(y_real, y_pred),
    }

comparacao = pd.DataFrame({
    "media_movel": metricas(y_teste, pred_baseline),
    "regressao": metricas(y_teste, pred_reg),
}).T.round({"mae": 0, "mape": 1, "r2": 3})
comparacao

,mae,mape,r2
media_movel,46289.0,18.9,-0.891
regressao,51850.0,19.8,-0.884


Resultado honesto: **no teste, o baseline ganha por pouco** (MAE 46,3k contra 51,9k;
MAPE 18,9% contra 19,8%) e os dois tem R2 negativo.

O que aconteceu: 2026 abriu num patamar bem acima do que 2025 sugeria (jan/2026 = 293k
contra 186k de jan/2025; fev/2026 = 331k contra 180k). A regressao paga o preco das
dummies de jan-jun estimadas com uma observacao so - a previsao dela para esses meses e
essencialmente "2025 + tendencia", e a tendencia, diluida em 18 meses, nao acompanha o
salto de nivel. A media movel, que so olha os 3 meses anteriores, se adapta ao patamar
novo mais rapido. E a vantagem classica do baseline reativo quando ha quebra de nivel.

R2 negativo nos dois significa que a media simples dos proprios 6 meses de teste teria
errado menos que qualquer um dos modelos. Com n=6 e um abril atipico (193k, menor mes
desde mai/2025) isso nao e vergonha, mas e o numero mais importante da tabela: nenhum
dos dois preve bem mes a mes, e a pagina de Previsao precisa dizer isso em vez de
esconder.

Por que a projecao ainda assim usa a regressao, e nao o baseline vencedor: media movel
projetada para frente e estruturalmente cega a sazonalidade - realimentada, ela converge
para uma reta e jamais anteciparia o pico de novembro, que e o evento comercial mais
importante do ano. A regressao carrega a dummy de novembro. Uma diferenca de MAE de 12%
relativos, medida em 6 observacoes, e pequena demais para descartar o unico dos dois
modelos capaz de dizer "novembro vai ser maior". Decisao registrada em
`docs/decisoes.md`.

In [7]:
# modelo final: re-treina com os 24 meses (o split serviu para medir; jogar fora
# os 6 meses mais recentes na hora de projetar seria desperdicio de dado)
reg_full = LinearRegression().fit(X, y)

futuro = pd.DataFrame({"ano_mes": [f"2026-{m:02d}" for m in range(7, 13)]})
futuro["t"] = np.arange(len(df), len(df) + len(futuro))
futuro["mes"] = futuro["ano_mes"].str[5:].astype(int)
X_fut = pd.get_dummies(futuro[["t", "mes"]], columns=["mes"], prefix="m",
                       drop_first=True).reindex(columns=X.columns, fill_value=False)
futuro["previsto"] = reg_full.predict(X_fut)

# banda: +-1.96 desvios do residuo FORA da amostra (teste), nao do treino -
# residuo de treino subestima o erro real
resid_teste = y_teste - pred_reg
banda = 1.96 * resid_teste.std(ddof=1)
futuro["banda_inf"] = futuro["previsto"] - banda
futuro["banda_sup"] = futuro["previsto"] + banda
futuro[["ano_mes", "previsto", "banda_inf", "banda_sup"]].round(0)

,ano_mes,previsto,banda_inf,banda_sup
0,2026-07,321339.0,207476.0,435202.0
1,2026-08,373141.0,259278.0,487004.0
2,2026-09,370922.0,257059.0,484785.0
3,2026-10,337312.0,223449.0,451175.0
4,2026-11,540511.0,426648.0,654374.0
5,2026-12,431687.0,317824.0,545550.0


Previsao do proximo mes (jul/2026): **~321k**, banda de +-114k (1,96 desvios do residuo
de teste). E o numero que abre a pagina de Previsao, e ele so e util acompanhado das
limitacoes:

- **24 observacoes.** Todo coeficiente aqui carrega incerteza grande; a banda cobre
  residuo de previsao, nao incerteza de parametro, entao tende ao otimismo.
- **Banda constante no horizonte.** O erro real cresce quanto mais longe se projeta:
  dez/2026 e mais incerto que jul/2026 e a banda nao diferencia os dois.
- **O MAPE esconde assimetria.** Um erro de 40k em abril (base 193k) pesa 21%; o mesmo
  erro em novembro (base 500k) pesa 8%. O MAPE medio parece equilibrado enquanto o
  erro absoluto de novembro tende a ser o maior do ano - para caixa, o que importa e
  o absoluto.
- **Sazonalidade media.** As dummies aprendem a media dos dois novembros; se a Black
  Friday de 2026 crescer como a de 2025 (+49% sobre 2024), o modelo subestima.
- **Nenhuma variavel externa** (marketing, preco, calendario promocional). O modelo
  descreve o passado da propria serie, nada mais.

TODO: com um terceiro ano de dado, testar a regressao em log(faturamento) - estabiliza
a variancia dos picos e transforma a tendencia em crescimento percentual, que e como o
negocio pensa.